# NB-06 — めぐ指数: 有効性検証レポート

**目的**: めぐ指数の総合的な予測有効性を多角的に検証し、最終レポートを生成する

**検証観点**:
1. めぐ指数と着順の相関（スピアマン・ケンドール）
2. 指数デシル別の勝率・複勝率（単調増加するか）
3. 指数1位馬の単回収率・複回収率（期待値 > 100 か）
4. 指数上位N頭に絞った場合の回収率シミュレーション
5. 距離帯・馬場種別・クラス別の有効性差
6. モデル再推定頻度の推奨（係数の経時安定性）

**最終出力**: 有効性サマリーレポート (`effectiveness_report.md`)

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/keiba-vpn')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy import stats

INPUT_NB01 = Path('output/nb01')
INPUT_NB02 = Path('output/nb02')
INPUT_NB05 = Path('output/nb05')
OUTPUT_DIR = Path('output/nb06')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_index = pd.read_parquet(INPUT_NB02 / 'megu_index_results.parquet')
df_race  = pd.read_parquet(INPUT_NB01 / 'megu_dataset.parquet')

df = df_index.merge(
    df_race[['race_id', 'horse_id', 'finish_pos', 'distance', 'surface',
             'track_condition', 'race_date', 'course', 'grade']].drop_duplicates(),
    on=['race_id', 'horse_id'], how='inner'
)
df['race_date'] = pd.to_datetime(df['race_date'])

# オッズデータが利用可能な場合（回収率計算用）
from src.db.session import get_session, init_engine
from sqlalchemy import text

init_engine()

try:
    with get_session() as session:
        df_odds = pd.read_sql(text("""
            SELECT DISTINCT ON (race_id, horse_id)
                race_id, horse_id, odds_value as win_odds
            FROM race_odds_snapshot
            WHERE snapshot_type = 'WIN'
            ORDER BY race_id, horse_id, snapshot_at DESC
        """), session.bind)
    df = df.merge(df_odds, on=['race_id', 'horse_id'], how='left')
    print(f'オッズデータ: {df["win_odds"].notna().mean()*100:.1f}% マッチ')
except Exception as e:
    print(f'オッズ取得失敗（回収率計算をスキップ）: {e}')
    df['win_odds'] = np.nan

print(f'統合データ: {len(df):,} 行, ユニークレース: {df["race_id"].nunique():,}')

## 1. 着順との相関

In [ ]:
df_valid = df.dropna(subset=['megu_index', 'finish_pos']).copy()

# レース内スピアマン相関の分布
spearman_by_race = df_valid.groupby('race_id').apply(
    lambda g: stats.spearmanr(g['megu_index'], g['finish_pos'])[0]
    if len(g) >= 4 else np.nan
).dropna()

print(f'レース内スピアマン相関（めぐ指数 vs 着順）:')
print(f'  平均: {spearman_by_race.mean():.4f}')
print(f'  中央値: {spearman_by_race.median():.4f}')
print(f'  負相関（期待: 指数高いほど着順良いなら負）の割合: {(spearman_by_race < 0).mean()*100:.1f}%')
print()
print('※ めぐ指数が高い = 速い → 着順（1=1着）が小さい → 期待符号は負')

spearman_by_race.hist(bins=40, color='steelblue', alpha=0.8)
plt.axvline(spearman_by_race.mean(), color='red', linestyle='--', label=f'平均={spearman_by_race.mean():.3f}')
plt.xlabel('スピアマン相関係数')
plt.title('レース内スピアマン相関の分布（めぐ指数 vs 着順）')
plt.legend()
plt.savefig(OUTPUT_DIR / 'spearman_distribution.png', dpi=120)
plt.show()

## 2. 指数デシル別勝率・複勝率

In [ ]:
# 全頭に対するデシル（指数の相対的な高さ）
df_valid['megu_decile'] = pd.qcut(df_valid['megu_index'], q=10, labels=False) + 1

decile_stats = df_valid.groupby('megu_decile').agg(
    n=('finish_pos', 'count'),
    win_rate=('finish_pos', lambda x: (x == 1).mean()),
    show_rate=('finish_pos', lambda x: (x <= 3).mean()),
    avg_pos=('finish_pos', 'mean'),
).reset_index()

print('=== めぐ指数デシル別 勝率・複勝率 ===')
print(decile_stats.to_string(index=False, float_format='{:.4f}'.format))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(decile_stats['megu_decile'], decile_stats['win_rate'], color='steelblue', alpha=0.8)
ax1.set_xlabel('めぐ指数デシル (1=低, 10=高)')
ax1.set_ylabel('勝率')
ax1.set_title('めぐ指数デシル別 勝率')
ax1.axhline(1/df_valid['finish_pos'].count() * df_valid['race_id'].nunique(), 
            color='red', linestyle='--', label='期待値')

ax2.bar(decile_stats['megu_decile'], decile_stats['show_rate'], color='coral', alpha=0.8)
ax2.set_xlabel('めぐ指数デシル (1=低, 10=高)')
ax2.set_ylabel('複勝率')
ax2.set_title('めぐ指数デシル別 複勝率')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'decile_winrate.png', dpi=120)
plt.show()

# 単調増加しているかチェック
is_mono_win  = all(decile_stats['win_rate'].iloc[i] <= decile_stats['win_rate'].iloc[i+1]
                   for i in range(len(decile_stats)-1))
is_mono_show = all(decile_stats['show_rate'].iloc[i] <= decile_stats['show_rate'].iloc[i+1]
                   for i in range(len(decile_stats)-1))
print(f'\n勝率の単調増加: {"✅" if is_mono_win else "⚠️ 部分的に逆転あり"}')
print(f'複勝率の単調増加: {"✅" if is_mono_show else "⚠️ 部分的に逆転あり"}')

## 3. 回収率シミュレーション

In [ ]:
if df_valid['win_odds'].notna().mean() > 0.3:
    # 指数1位馬を全レースで単勝購入した場合の回収率
    df_valid['score_rank'] = df_valid.groupby('race_id')['megu_index'].rank(ascending=False, method='min')

    strategies = [
        ('指数1位のみ',    df_valid[df_valid['score_rank'] == 1]),
        ('指数1〜2位',     df_valid[df_valid['score_rank'] <= 2]),
        ('指数1〜3位',     df_valid[df_valid['score_rank'] <= 3]),
        ('デシル10のみ',  df_valid[df_valid['megu_decile'] == 10]),
    ]

    print('=== 回収率シミュレーション（単勝: 100円購入）===')
    for name, subset in strategies:
        sub = subset.dropna(subset=['win_odds', 'finish_pos'])
        if len(sub) == 0:
            continue
        bets = len(sub)
        wins = (sub['finish_pos'] == 1).sum()
        payout = sub[sub['finish_pos'] == 1]['win_odds'].sum() * 100  # 100円賭け
        cost   = bets * 100
        roi    = payout / cost * 100 if cost > 0 else 0
        print(f'  {name}: 購入{bets}件, 的中{wins}件, 回収率={roi:.1f}%')
else:
    print('オッズデータ不足のためスキップ（期待値 > 100% かどうかは実データで確認）')

## 4. セグメント別有効性

In [ ]:
def segment_spearman(group):
    if len(group) < 50:
        return np.nan
    per_race = group.groupby('race_id').apply(
        lambda g: stats.spearmanr(g['megu_index'], g['finish_pos'])[0]
        if len(g) >= 4 else np.nan
    )
    return per_race.mean()

def distance_band_label(d):
    if d < 1500: return 'sprint'
    if d < 1800: return 'mile'
    if d < 2400: return 'middle'
    return 'long'

df_valid['dist_band'] = df_valid['distance'].apply(distance_band_label)

print('=== セグメント別 スピアマン相関（高いほど有効） ===')

# 距離帯別
print('\n距離帯別:')
for band in ['sprint', 'mile', 'middle', 'long']:
    sub = df_valid[df_valid['dist_band'] == band]
    r = segment_spearman(sub)
    print(f'  {band}: {r:.4f} (n={len(sub):,})')

# 馬場種別
print('\n馬場種別:')
for surface in ['芝', 'ダート']:
    sub = df_valid[df_valid['surface'] == surface]
    r = segment_spearman(sub)
    print(f'  {surface}: {r:.4f} (n={len(sub):,})')

# グレード別（グレードデータがある場合）
if 'grade' in df_valid.columns:
    print('\nグレード別（主要カテゴリ）:')
    grade_map = {'G1': 'G1', 'G2': 'G2', 'G3': 'G3', '': 'その他'}
    df_valid['grade_cat'] = df_valid['grade'].fillna('').map(
        lambda g: 'G1-3' if g in ('G1','G2','G3') else '条件戦'
    )
    for cat in ['G1-3', '条件戦']:
        sub = df_valid[df_valid['grade_cat'] == cat]
        r = segment_spearman(sub)
        print(f'  {cat}: {r:.4f} (n={len(sub):,})')

## 5. 係数の経時安定性（モデル再推定頻度の検討）

In [ ]:
import statsmodels.formula.api as smf

# 年別に β₁（ペース補正）の推定値を比較
df_valid['year'] = df_valid['race_date'].dt.year

STD_WEIGHT_MALE   = 55.0
STD_WEIGHT_FEMALE = 53.0

# 簡易検証: 年別の指数1位命中率の推移（係数安定性の proxy）
df_valid['score_rank'] = df_valid.groupby('race_id')['megu_index'].rank(ascending=False, method='min')

yearly = df_valid.groupby('year').apply(
    lambda g: pd.Series({
        'n_races': g['race_id'].nunique(),
        'top1_win_rate': (g[(g['score_rank'] == 1) & (g['finish_pos'] == 1)].shape[0] /
                          g[g['score_rank'] == 1].shape[0])
        if g[g['score_rank'] == 1].shape[0] > 0 else np.nan,
    })
).reset_index()

print('=== 年別 指数1位の勝率推移 ===')
print(yearly.to_string(index=False, float_format='{:.4f}'.format))

yearly['top1_win_rate'].plot(kind='bar', color='steelblue', alpha=0.8)
plt.xlabel('年')
plt.ylabel('指数1位の勝率')
plt.title('年別 指数1位命中率の推移')
plt.xticks(range(len(yearly)), yearly['year'].astype(int).tolist(), rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'yearly_stability.png', dpi=120)
plt.show()

# 安定性の判断
cv = yearly['top1_win_rate'].std() / yearly['top1_win_rate'].mean()
print(f'\n年別勝率の変動係数 (CV): {cv:.4f}')
if cv < 0.10:
    freq_recommendation = '年次（モデルは安定）'
elif cv < 0.20:
    freq_recommendation = '四半期（軽微な季節性あり）'
else:
    freq_recommendation = '月次（有意な経時変化あり）'

print(f'推奨再推定頻度: {freq_recommendation}')

## 6. 有効性サマリーレポート生成

In [ ]:
# 最終サマリーの生成
mean_spearman = spearman_by_race.mean()
neg_spearman_pct = (spearman_by_race < 0).mean() * 100
top_decile_winrate = decile_stats[decile_stats['megu_decile'] == 10]['win_rate'].values[0]
bot_decile_winrate = decile_stats[decile_stats['megu_decile'] == 1]['win_rate'].values[0]

report_lines = [
    '# めぐ指数 有効性検証レポート',
    '',
    '## 1. 着順との相関',
    f'- レース内スピアマン相関（平均）: **{mean_spearman:.4f}**',
    f'- 期待方向（負相関）の割合: **{neg_spearman_pct:.1f}%**',
    '',
    '## 2. デシル別勝率',
    f'- 上位デシル（10）の勝率: **{top_decile_winrate:.4f}**',
    f'- 下位デシル（1）の勝率: **{bot_decile_winrate:.4f}**',
    f'- 勝率単調増加: **{"✅" if is_mono_win else "⚠️"}**',
    f'- 複勝率単調増加: **{"✅" if is_mono_show else "⚠️"}**',
    '',
    '## 3. モデル再推定頻度',
    f'- 推奨: **{freq_recommendation}**',
    f'- 年別勝率変動係数 (CV): {cv:.4f}',
    '',
    '## 4. 結論',
]

if mean_spearman < -0.15 and neg_spearman_pct > 70 and is_mono_win:
    report_lines.append('✅ めぐ指数は着順に対して十分な予測有効性を示している')
elif mean_spearman < -0.10:
    report_lines.append('⚠️ 指数の方向性は正しいが、更なる補正変数の改善余地あり')
else:
    report_lines.append('❌ 有効性が不十分 — 統合回帰モデルの再検討が必要')

report_text = '\n'.join(report_lines)
print(report_text)

with open(OUTPUT_DIR / 'effectiveness_report.md', 'w', encoding='utf-8') as f:
    f.write(report_text)

print(f'\nレポートを保存: {OUTPUT_DIR / "effectiveness_report.md"}')

# 数値サマリーを保存
summary = {
    'mean_spearman': float(mean_spearman),
    'neg_spearman_pct': float(neg_spearman_pct),
    'top_decile_winrate': float(top_decile_winrate),
    'bot_decile_winrate': float(bot_decile_winrate),
    'is_monotone_win': bool(is_mono_win),
    'is_monotone_show': bool(is_mono_show),
    'yearly_cv': float(cv),
    'refit_recommendation': freq_recommendation,
}
import json
with open(OUTPUT_DIR / 'effectiveness_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('数値サマリーを保存しました')